In [ ]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo

ticker = "UNH"
# Exclude the current New York date so an intraday bar never becomes a close.
end_date = datetime.now(ZoneInfo("America/New_York")).date()
prices = yf.download(ticker, start="2022-01-01", end=end_date.isoformat(), auto_adjust=True)
if prices.empty:
    raise RuntimeError(f"No prices returned for {ticker}. Check the ticker and network connection.")

In [ ]:
close = prices["Close"].squeeze("columns").dropna().sort_index() if isinstance(prices["Close"], pd.DataFrame) else prices["Close"].dropna().sort_index()
close = close.rename(ticker)

In [ ]:
returns = close.pct_change().dropna()

In [ ]:
print(returns.head())

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12,8))

close.plot(ax=axes[0], title=f"{ticker} Closing Price")
returns.plot(ax=axes[1], title=f"{ticker} Daily Returns")

plt.tight_layout()
plt.show()

In [ ]:
prices_2023 = prices.loc["2023"]

In [ ]:
prices_2023

In [ ]:
Trend = prices.loc['2022-01-01':'2025-12-31', "Close"]

Trend.plot(figsize=(12,8))

plt.title(f"{ticker} adjusted closing price")
plt.xlabel("date")
plt.ylabel("Adjusted Close Price (USD)")
plt.grid(True)
plt.show()

In [ ]:
month_end_close = close.resample("ME").last()
print(month_end_close.head())

In [ ]:
monthly_mean_close = close.resample("ME").mean()
print(monthly_mean_close.head())

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

close.plot(
    ax=ax,
    label="Daily Close",
    alpha=0.5
)
monthly_mean_close.plot(
    ax=ax,
    label="Monthly Mean Close",
    marker="o"
)
ax.set_title(f"{ticker} daily and monthly mean closing prices")
ax.set_xlabel("Date")
ax.set_ylabel("Adjusted Price (USD)")
ax.legend()
ax.grid(True)
plt.show()

## Next trading session forecast
Estimate the next adjusted close from returns and volatility available at the latest included close. The backtest refits the model at each forecast date and reserves the final 20% of labeled observations for evaluation.

In [ ]:
import numpy as np

daily = close.pct_change()
features = pd.DataFrame({
    "daily_return": daily,
    "weekly_return": close.pct_change(5),
    "monthly_return": close.pct_change(20),
    "monthly_volatility": daily.rolling(20).std(),
}).dropna()
next_return = daily.shift(-1).reindex(features.index)
labeled = features.loc[next_return.dropna().index]
if len(labeled) < 100:
    raise ValueError("Need at least 100 labeled sessions after the feature warm-up.")

def forecast_return(history, observed_returns, current):
    # Standardize using training history only, then fit ridge with a fixed penalty.
    center = history.mean()
    scale = history.std(ddof=0).replace(0, 1)
    x = ((history - center) / scale).to_numpy()
    y = observed_returns.to_numpy()
    weights = np.linalg.solve(x.T @ x + 10 * np.eye(x.shape[1]), x.T @ (y - y.mean()))
    return float(y.mean() + ((current - center) / scale).to_numpy() @ weights)

In [ ]:
split = int(0.8 * len(labeled))
records = []
for i in range(split, len(labeled)):
    history = labeled.iloc[:i]
    estimate = forecast_return(history, next_return.loc[history.index], labeled.iloc[i])
    today = close.loc[labeled.index[i]]
    records.append((today * (1 + estimate), today))

target_dates = close.index[close.index.get_indexer(labeled.index[split:]) + 1]
backtest = pd.DataFrame(records, index=target_dates, columns=["Model", "Previous close"])
backtest["Actual"] = close.loc[target_dates]
errors = backtest[["Model", "Previous close"]].sub(backtest["Actual"], axis=0)
print(f"{len(backtest)} forecasts from {target_dates[0]:%Y-%m-%d} through {target_dates[-1]:%Y-%m-%d}")
print(pd.DataFrame({"MAE (USD)": errors.abs().mean(), "RMSE (USD)": errors.pow(2).mean().pow(0.5)}).round(3))
backtest.tail(60).plot(figsize=(12, 5), title=f"{ticker} forecasts by target session", ylabel="Adjusted close (USD)")
plt.show()

In [ ]:
latest = features.iloc[-1]
estimate = forecast_return(labeled, next_return.loc[labeled.index], latest)
last_close = close.loc[latest.name]
print(f"{ticker} adjusted close on {latest.name:%Y-%m-%d}: ${last_close:,.2f}")
print(f"Forecast for the following trading session: ${last_close * (1 + estimate):,.2f} ({estimate:+.2%})")

## 뉴스 감성이 다음 거래일 예측에 도움이 될까?
Alpaca에서 같은 종목의 영어 기사 제목과 요약을 가져옵니다. 실행 전 Jupyter 환경에 APCA_API_KEY_ID와 APCA_API_SECRET_KEY를 설정하세요. 키가 없으면 위쪽의 주가 전용 예측까지 사용할 수 있습니다.

각 거래일 뉴욕 정오까지 공개된 뉴스와 그날의 종가를 사용해 다음 거래일 종가를 예측합니다. 정오 이후 뉴스는 다음 거래일의 입력으로 넘어갑니다. 정오 기준은 조기 폐장일에도 마감 이후 기사가 섞이지 않도록 정했습니다. 수정된 기사는 마지막 수정 시각부터 사용합니다.

In [ ]:
import os
import re
import requests

def fetch_news(symbol, start, end):
    headers = {
        "APCA-API-KEY-ID": os.environ["APCA_API_KEY_ID"],
        "APCA-API-SECRET-KEY": os.environ["APCA_API_SECRET_KEY"],
    }
    params = dict(symbols=symbol, start=start.isoformat(), end=end.isoformat(), limit=50, sort="asc")
    articles = {}
    while True:
        response = requests.get("https://data.alpaca.markets/v1beta1/news",
                                headers=headers, params=params, timeout=30)
        response.raise_for_status()
        page = response.json()
        for article in page["news"]:
            articles[article["id"]] = article
        token = page.get("next_page_token")
        if not token:
            return list(articles.values())
        params["page_token"] = token

In [ ]:
# 간단한 금융 표현 사전입니다. 일치하는 표현이 없으면 점수는 0입니다.
positive = r"\b(?:beats? (?:estimates|expectations)|raises? (?:guidance|outlook)|upgrades?|upgraded|outperform|record profit|surges?)\b"
negative = r"\b(?:miss(?:es)? (?:estimates|expectations)|cuts? (?:guidance|outlook)|downgrades?|downgraded|underperform|lawsuits?|investigations?|plunges?)\b"

def sentiment(text):
    up = len(re.findall(positive, text, re.I))
    down = len(re.findall(negative, text, re.I))
    return (up - down) / max(up + down, 1)

def news_features(articles, sessions, cutoffs):
    rows = []
    for article in articles:
        available = pd.Timestamp(article.get("updated_at") or article["created_at"])
        i = cutoffs.searchsorted(available)
        if i < len(sessions):
            text = " ".join([article.get("headline") or "", article.get("summary") or ""]).strip()
            if text:
                rows.append((sessions[i], sentiment(text)))
    scores = pd.DataFrame(rows, columns=["session", "score"])
    return (scores.groupby("session")["score"].agg(news_mean="mean", news_count="size")
            .reindex(sessions).fillna(0))

In [ ]:
sessions = close.index
cutoffs = (sessions.tz_localize(None).normalize() + pd.Timedelta(hours=12)).tz_localize("America/New_York")
news_inputs = None
if all(os.getenv(key) for key in ["APCA_API_KEY_ID", "APCA_API_SECRET_KEY"]):
    articles = fetch_news(ticker, cutoffs[0] - pd.Timedelta(days=7), cutoffs[-1])
    if articles:
        news_inputs = news_features(articles, sessions, cutoffs)
    else:
        print("조회 기간에 기사가 없습니다. 주가 전용 결과를 사용하세요.")
else:
    print("뉴스 분석을 실행하려면 Jupyter 환경에 Alpaca API 키 두 개를 설정하세요.")

In [ ]:
if news_inputs is not None:
    combined = features.join(news_inputs)
    predictions = []
    for i in range(split, len(labeled)):
        history = combined.loc[labeled.index[:i]]
        date = labeled.index[i]
        r = forecast_return(history, next_return.loc[history.index], combined.loc[date])
        predictions.append(close.loc[date] * (1 + r))

    comparison = backtest.copy()
    comparison["Price + news"] = predictions
    err = comparison.drop(columns="Actual").sub(comparison["Actual"], axis=0)
    print(pd.DataFrame({"MAE (USD)": err.abs().mean(), "RMSE (USD)": err.pow(2).mean().pow(0.5)}).round(3))
    print("뉴스 추가 시 MAE 감소액 (양수면 개선):", round(err["Model"].abs().mean() - err["Price + news"].abs().mean(), 3))
    history = combined.loc[labeled.index]
    r = forecast_return(history, next_return.loc[history.index], combined.iloc[-1])
    print(f"{ticker} {combined.index[-1]:%Y-%m-%d} 기준 다음 거래일 예상 수정 종가: ${last_close * (1 + r):,.2f}")
    print(f"최근 입력 기사 {int(combined.news_count.iloc[-1])}건, 평균 감성 {combined.news_mean.iloc[-1]:+.2f}")